# Train a PneumoniaMNIST CNN

This notebook is an educational starting point for a small binary image-classification project. It does **not** produce a clinically validated diagnostic system.

The dataset contains 28×28 grayscale images. Run cells in order.

In [ ]:
!pip -q install medmnist scikit-learn matplotlib

In [ ]:
import copy
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import PneumoniaMNIST
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Load the dataset

MedMNIST provides separate training, validation, and test splits. Keep the test split for final evaluation rather than tuning the model against it.

In [ ]:
transform = transforms.ToTensor()

train_data = PneumoniaMNIST(split="train", download=True, transform=transform)
val_data = PneumoniaMNIST(split="val", download=True, transform=transform)
test_data = PneumoniaMNIST(split="test", download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
val_loader = DataLoader(val_data, batch_size=256, shuffle=False)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

image, label = train_data[0]
plt.imshow(image.squeeze(), cmap="gray")
plt.title(f"Dataset label: {int(np.asarray(label).item())}")
plt.axis("off")
plt.show()

## 2. Define a small CNN

In [ ]:
class PneumoniaCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = PneumoniaCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(model)

## 3. Train and validate

Validation loss is tracked so you can see whether training is improving. Five epochs is only a starter setting; it is not evidence that the model is reliable.

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images = images.to(device)
        labels = torch.as_tensor(labels).view(-1).long().to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total

best_val_loss = float("inf")
best_weights = copy.deepcopy(model.state_dict())
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(5):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1}/5 | "
        f"train loss {train_loss:.4f}, acc {train_acc:.3f} | "
        f"val loss {val_loss:.4f}, acc {val_acc:.3f}"
    )

model.load_state_dict(best_weights)

In [ ]:
plt.plot(history["train_loss"], label="Train loss")
plt.plot(history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 4. Evaluate once on the test split

Accuracy alone is not enough for medical-image research. Review the per-class precision/recall and confusion matrix, and discuss dataset limitations in the project report.

In [ ]:
model.eval()
all_predictions, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = torch.as_tensor(labels).view(-1).long()
        predictions = model(images).argmax(1).cpu()
        all_predictions.extend(predictions.numpy().tolist())
        all_labels.extend(labels.numpy().tolist())

print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1],
    target_names=["Normal", "Pneumonia-associated"],
    zero_division=0
))
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(all_labels, all_predictions, labels=[0, 1]))

## 5. Save model weights

Download the resulting file from the Colab file browser and put it in your local project's `models/` folder as `pneumonia_cnn.pth`. The Streamlit demo expects this exact path.

In [ ]:
torch.save(model.state_dict(), "pneumonia_cnn.pth")
print("Saved pneumonia_cnn.pth")